## 1. Mount Google Drive
Connect your Google Drive where the project repository is located.


In [1]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Configuration & Paths
Define project paths and training hyperparameters. You only need to modify `PROJECT_ROOT` if your directory structure differs.


In [2]:
import os

# ==============================================================================
# PROJECT & RUNTIME CONFIGURATION
# ==============================================================================
# Base directory where Architectural-Styles-LoRA repository is located on Google Drive:
PROJECT_ROOT = "/content/drive/MyDrive/Architectural-Styles-LoRA"

EXPERIMENT_NAME = "brutalism_sdxl_lora_v1"
MODEL_ID = "stabilityai/stable-diffusion-xl-base-1.0"
VAE_ID = "madebyollin/sdxl-vae-fp16-fix"

# Training Hyperparameters
RESOLUTION = 768            # Conservative 768px for pilot stability and VRAM headroom
BATCH_SIZE = 1              # Batch size per device
GRADIENT_ACCUMULATION = 4   # Effective batch size = 4
LEARNING_RATE = 1e-4        # Learning rate for UNet LoRA
LORA_RANK = 16              # Rank 16 prevents memorization on small 22-image dataset
LORA_ALPHA = 16             # Standard rank:alpha ratio 1:1
NUM_EPOCHS = 15             # 15 epochs across 22 images
MAX_TRAIN_STEPS = 330       # Total training steps (Set to 50 for quick dry run)
SEED = 42

# Directory Paths
DATA_DIR = os.path.join(PROJECT_ROOT, "data", "clean")
IMAGES_DIR = os.path.join(DATA_DIR, "images", "brutalism")
TRAIN_CSV = os.path.join(DATA_DIR, "metadata", "train.csv")
VAL_CSV = os.path.join(DATA_DIR, "metadata", "val.csv")
TEST_CSV = os.path.join(DATA_DIR, "metadata", "test.csv")

OUTPUT_DIR = os.path.join(PROJECT_ROOT, "outputs", EXPERIMENT_NAME)
BASELINE_DIR = os.path.join(OUTPUT_DIR, "baseline")
LORA_DIR = os.path.join(OUTPUT_DIR, "lora")
VAL_DIR = os.path.join(OUTPUT_DIR, "validation")
EVAL_DIR = os.path.join(OUTPUT_DIR, "evaluation")
FINAL_DIR = os.path.join(OUTPUT_DIR, "final")

for d in [OUTPUT_DIR, BASELINE_DIR, LORA_DIR, VAL_DIR, EVAL_DIR, FINAL_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"Project Root     : {PROJECT_ROOT}")
print(f"Output Directory : {OUTPUT_DIR}")
print(f"Max Train Steps  : {MAX_TRAIN_STEPS}")



Project Root     : /content/drive/MyDrive/Architectural-Styles-LoRA
Output Directory : /content/drive/MyDrive/Architectural-Styles-LoRA/outputs/brutalism_sdxl_lora_v1
Max Train Steps  : 330


## 3. GPU Verification & Guardrails
Audit GPU model and available VRAM. Halts execution if GPU is missing or VRAM is insufficient (<11 GB).


In [3]:
import sys
import torch

print("=== Colab Environment & Hardware Verification ===")
print(f"Python version : {sys.version.split()[0]}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available : {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError("CRITICAL ERROR: No GPU detected! Go to Runtime -> Change runtime type -> Select T4, L4, or A100 GPU.")

gpu_name = torch.cuda.get_device_name(0)
props = torch.cuda.get_device_properties(0)
total_vram_gb = props.total_memory / (1024**3)

print(f"GPU Model      : {gpu_name}")
print(f"Total VRAM     : {total_vram_gb:.2f} GB")
print(f"CUDA Capability: {props.major}.{props.minor}")

if total_vram_gb < 11.0:
    raise RuntimeError(
        f"INSUFFICIENT VRAM: {gpu_name} has only {total_vram_gb:.2f} GB VRAM. "
        "SDXL LoRA training requires at least 11 GB VRAM. "
        "Please select a T4 (16 GB), L4 (24 GB), or A100 (40 GB) runtime in Google Colab."
    )
else:
    print(f"SUCCESS: {gpu_name} with {total_vram_gb:.2f} GB VRAM is sufficient for SDXL LoRA pilot training!")



=== Colab Environment & Hardware Verification ===
Python version : 3.13.15
PyTorch version: 2.11.0+cu128
CUDA available : True
GPU Model      : Tesla T4
Total VRAM     : 14.56 GB
CUDA Capability: 7.5
SUCCESS: Tesla T4 with 14.56 GB VRAM is sufficient for SDXL LoRA pilot training!


## 4. Install Pinned Dependencies
Installs verified versions of Hugging Face Diffusers, Accelerate, PEFT, and bitsandbytes.


In [4]:
!pip install -q \
    "torch>=2.1.0" \
    "torchvision>=0.16.0" \
    "diffusers>=0.30.0" \
    "transformers>=4.44.0" \
    "accelerate>=0.33.0" \
    "peft>=0.12.0" \
    "safetensors>=0.4.0" \
    "bitsandbytes>=0.43.0" \
    "scipy" \
    "scikit-learn" \
    "pandas" \
    "pillow"



## 5. Verify Installed Dependencies
Verify package imports and version compatibility.


In [5]:
import importlib

packages = [
    "torch", "torchvision", "diffusers", "transformers",
    "accelerate", "peft", "safetensors", "bitsandbytes"
]

print("=== Installed Package Versions ===")
for pkg in packages:
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, "__version__", "available")
        print(f"  {pkg:15}: {ver}")
    except ImportError as e:
        print(f"  {pkg:15}: ERROR - {e}")



=== Installed Package Versions ===
  torch          : 2.11.0+cu128
  torchvision    : 0.26.0+cu128
  diffusers      : 0.40.0
  transformers   : 5.16.1
  accelerate     : 1.14.0
  peft           : 0.20.0
  safetensors    : 0.8.0
  bitsandbytes   : 0.50.2


## 6. Dataset Verification
Checks that exactly 22 training images, 3 validation images, and 3 test images exist, open correctly, and meet resolution standards.


In [6]:
import os
import pandas as pd
from PIL import Image

print("=== Verifying Brutalism Pilot Dataset ===")

for path_name, path_val in [("Train CSV", TRAIN_CSV), ("Val CSV", VAL_CSV), ("Test CSV", TEST_CSV)]:
    if not os.path.exists(path_val):
        raise FileNotFoundError(f"Missing required metadata file: {path_val}")

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

brut_train = train_df[train_df["style"] == "Brutalism architecture"].copy()
brut_val = val_df[val_df["style"] == "Brutalism architecture"].copy()
brut_test = test_df[test_df["style"] == "Brutalism architecture"].copy()

print(f"Brutalism Training records   : {len(brut_train)}")
print(f"Brutalism Validation records : {len(brut_val)}")
print(f"Brutalism Test records       : {len(brut_test)}")

if len(brut_train) != 22:
    raise ValueError(f"Expected exactly 22 training images, but found {len(brut_train)}!")
if len(brut_val) != 3:
    raise ValueError(f"Expected exactly 3 validation images, but found {len(brut_val)}!")
if len(brut_test) != 3:
    raise ValueError(f"Expected exactly 3 test images, but found {len(brut_test)}!")

all_brut = pd.concat([brut_train, brut_val, brut_test], ignore_index=True)
for idx, row in all_brut.iterrows():
    img_file = os.path.join(IMAGES_DIR, f"{row['image_id']}.jpg")
    if not os.path.exists(img_file):
        raise FileNotFoundError(f"Missing image on disk: {img_file}")
    with Image.open(img_file) as im:
        if im.mode != "RGB":
            raise ValueError(f"Image {img_file} is not RGB: {im.mode}")
        if min(im.size) < 768:
            raise ValueError(f"Image {img_file} shortest side is {min(im.size)}px (< 768px)")
        if not str(row["caption"]).strip():
            raise ValueError(f"Image {row['image_id']} has empty caption!")

print("SUCCESS: All 28 Brutalism images verified on disk (22 train, 3 val, 3 test), all RGB >= 768px.")



=== Verifying Brutalism Pilot Dataset ===
Brutalism Training records   : 22
Brutalism Validation records : 3
Brutalism Test records       : 3
SUCCESS: All 28 Brutalism images verified on disk (22 train, 3 val, 3 test), all RGB >= 768px.


## 7. Load Base SDXL 1.0 Pipeline
Loads base SDXL 1.0 with the FP16-safe VAE to prevent NaN / black-image artifacts.


In [7]:
import torch
from diffusers import StableDiffusionXLPipeline, AutoencoderKL

print(f"Loading Base SDXL Pipeline ({MODEL_ID}) and VAE ({VAE_ID})...")

vae = AutoencoderKL.from_pretrained(
    VAE_ID,
    torch_dtype=torch.float16
)

pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_ID,
    vae=vae,
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True
).to("cuda")

pipe.enable_attention_slicing()
print("Base SDXL 1.0 pipeline successfully loaded in FP16!")



Loading Base SDXL Pipeline (stabilityai/stable-diffusion-xl-base-1.0) and VAE (madebyollin/sdxl-vae-fp16-fix)...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Base SDXL 1.0 pipeline successfully loaded in FP16!


## 8. Baseline SDXL Generation (Before LoRA)
Generates baseline images using predefined evaluation prompts and fixed seeds for direct before/after comparison.


In [8]:
import os
import pandas as pd
import torch

print("=== Generating Baseline SDXL 1.0 Images (Before LoRA) ===")

eval_prompts_file = os.path.join(PROJECT_ROOT, "evaluation", "brutalism_prompts.txt")
with open(eval_prompts_file, "r", encoding="utf-8") as f:
    eval_lines = [line.strip() for line in f if line.strip() and not line.startswith("#")]

# Select top 6 representative evaluation prompts
baseline_prompts = eval_lines[:6]
baseline_records = []

for idx, prompt_text in enumerate(baseline_prompts):
    clean_prompt = prompt_text.split(". ", 1)[-1] if ". " in prompt_text[:4] else prompt_text
    seed = SEED + idx
    generator = torch.Generator(device="cuda").manual_seed(seed)

    print(f"Generating Baseline [{idx+1}/{len(baseline_prompts)}] Seed={seed}: {clean_prompt[:60]}...")
    image = pipe(
        prompt=clean_prompt,
        num_inference_steps=30,
        guidance_scale=7.5,
        generator=generator,
        height=RESOLUTION,
        width=RESOLUTION
    ).images[0]

    out_name = f"baseline_eval_{idx+1:02d}.jpg"
    out_path = os.path.join(BASELINE_DIR, out_name)
    image.save(out_path, "JPEG", quality=95)

    baseline_records.append({
        "prompt_id": f"BASE_{idx+1:02d}",
        "prompt": clean_prompt,
        "seed": seed,
        "output_path": out_path
    })

baseline_df = pd.DataFrame(baseline_records)
baseline_df.to_csv(os.path.join(OUTPUT_DIR, "baseline_results.csv"), index=False)
print(f"Saved {len(baseline_records)} baseline images to {BASELINE_DIR}")



=== Generating Baseline SDXL 1.0 Images (Before LoRA) ===
Generating Baseline [1/6] Seed=42: Brutalist civic building with monumental exposed concrete ma...


  0%|          | 0/30 [00:00<?, ?it/s]

Generating Baseline [2/6] Seed=43: Brutalist art museum with monolithic raw concrete walls, sha...


  0%|          | 0/30 [00:00<?, ?it/s]

Generating Baseline [3/6] Seed=44: Brutalist university lecture hall complex with tiered board-...


...

Generating Baseline [5/6] Seed=46: Brutalist cultural center featuring heavy cantilevered concr...


  0%|          | 0/30 [00:00<?, ?it/s]

Generating Baseline [6/6] Seed=47: Brutalist municipal public library with exposed waffle slab ...


  0%|          | 0/30 [00:00<?, ?it/s]

Saved 6 baseline images to /content/drive/MyDrive/Architectural-Styles-LoRA/outputs/brutalism_sdxl_lora_v1/baseline


In [9]:
from PIL import Image
import matplotlib.pyplot as plt
import glob
import os
import math

baseline_dir = "/content/drive/MyDrive/Architectural-Styles-LoRA/outputs/brutalism_sdxl_lora_v1/baseline"

images = sorted(
    glob.glob(os.path.join(baseline_dir, "*.png")) +
    glob.glob(os.path.join(baseline_dir, "*.jpg")) +
    glob.glob(os.path.join(baseline_dir, "*.jpeg"))
)

print(f"Found {len(images)} baseline images")

cols = 3
rows = math.ceil(len(images) / cols)

fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows))
axes = axes.flatten() if hasattr(axes, "flatten") else [axes]

for ax, img_path in zip(axes, images):
    img = Image.open(img_path)
    ax.imshow(img)
    ax.set_title(os.path.basename(img_path), fontsize=10)
    ax.axis("off")

# Hide unused cells
for ax in axes[len(images):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

Found 6 baseline images


In [11]:
import torch
import diffusers
import transformers
import peft
import accelerate

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("Diffusers:", diffusers.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)

try:
    import torchao
    print("torchao:", torchao.__version__)
except ImportError:
    print("torchao: not installed — OK")

PyTorch: 2.11.0+cu128
CUDA: True
Diffusers: 0.40.0
Transformers: 5.16.1
PEFT: 0.20.0
torchao: not installed — OK


---
# 9. START TRAINING HERE
Execute the cell below to start the SDXL LoRA pilot training run.

- **Training Images**: 22 curated Brutalism photographs
- **Resolution**: 768×768
- **Rank / Alpha**: 16 / 16
- **Batch Size / Acc**: 1 / 4 (Effective batch size = 4)
- **Max Steps**: 330 steps (~15–20 min on Colab T4)
---



In [16]:
import os
import urllib.request

# Unload baseline pipeline if it exists
if "pipe" in globals():
    del pipe

import gc
gc.collect()
torch.cuda.empty_cache()

print("GPU memory cleared.")

# Download official Hugging Face diffusers SDXL LoRA training script
script_url = "https://raw.githubusercontent.com/huggingface/diffusers/v0.30.0/examples/text_to_image/train_text_to_image_lora_sdxl.py"
train_script = os.path.join(PROJECT_ROOT, "scripts", "train_text_to_image_lora_sdxl.py")
os.makedirs(os.path.dirname(train_script), exist_ok=True)

if not os.path.exists(train_script):
    print("Downloading train_text_to_image_lora_sdxl.py...")
    urllib.request.urlretrieve(script_url, train_script)
    print("Download complete.")

# Prepare training metadata in image directory format for diffusers
# (diffusers expects metadata.csv or metadata.jsonl inside the image folder)
brut_train_diffusers = brut_train[["image_id", "caption"]].copy()
brut_train_diffusers["file_name"] = brut_train_diffusers["image_id"] + ".jpg"
meta_csv_path = os.path.join(IMAGES_DIR, "metadata.csv")
brut_train_diffusers[["file_name", "caption"]].to_csv(meta_csv_path, index=False)
print(f"Prepared training metadata at {meta_csv_path}")

# Construct accelerate command
cmd = f"""accelerate launch --mixed_precision="fp16" "{train_script}" \
    --pretrained_model_name_or_path="{MODEL_ID}" \
    --pretrained_vae_model_name_or_path="{VAE_ID}" \
    --train_data_dir="{IMAGES_DIR}" \
    --output_dir="{FINAL_DIR}" \
    --image_column="image" \
    --caption_column="caption" \
    --resolution={RESOLUTION} \
    --random_flip \
    --train_batch_size={BATCH_SIZE} \
    --gradient_accumulation_steps={GRADIENT_ACCUMULATION} \
    --max_train_steps={MAX_TRAIN_STEPS} \
    --learning_rate={LEARNING_RATE} \
    --lr_scheduler="constant_with_warmup" \
    --lr_warmup_steps=30 \
    --rank={LORA_RANK} \
    --seed={SEED} \
    --mixed_precision="fp16" \
    --gradient_checkpointing \
    --checkpointing_steps=110 \
    --checkpoints_total_limit=3
"""

print("Executing LoRA training command:")
print(cmd)
!{cmd}



GPU memory cleared.
Prepared training metadata at /content/drive/MyDrive/Architectural-Styles-LoRA/data/clean/images/brutalism/metadata.csv
Executing LoRA training command:
accelerate launch --mixed_precision="fp16" "/content/drive/MyDrive/Architectural-Styles-LoRA/scripts/train_text_to_image_lora_sdxl.py"     --pretrained_model_name_or_path="stabilityai/stable-diffusion-xl-base-1.0"     --pretrained_vae_model_name_or_path="madebyollin/sdxl-vae-fp16-fix"     --train_data_dir="/content/drive/MyDrive/Architectural-Styles-LoRA/data/clean/images/brutalism"     --output_dir="/content/drive/MyDrive/Architectural-Styles-LoRA/outputs/brutalism_sdxl_lora_v1/final"     --image_column="image"     --caption_column="caption"     --resolution=768     --random_flip     --train_batch_size=1     --gradient_accumulation_steps=4     --max_train_steps=330     --learning_rate=0.0001     --lr_scheduler="constant_with_warmup"     --lr_warmup_steps=30     --rank=16     --seed=42     --mixed_precision="fp16"  

## 10. Checkpoint Inspection
Inspect saved LoRA weights and verify `pytorch_lora_weights.safetensors`.


In [17]:
import os

print(f"=== Inspecting Output Directory: {FINAL_DIR} ===")
for root, dirs, files in os.walk(FINAL_DIR):
    for f in files:
        if f.endswith((".safetensors", ".bin", ".json")):
            fpath = os.path.join(root, f)
            size_mb = os.path.getsize(fpath) / (1024 * 1024)
            print(f"  {os.path.relpath(fpath, FINAL_DIR)} ({size_mb:.2f} MB)")



=== Inspecting Output Directory: /content/drive/MyDrive/Architectural-Styles-LoRA/outputs/brutalism_sdxl_lora_v1/final ===
  pytorch_lora_weights.safetensors (88.75 MB)
  checkpoint-110/pytorch_lora_weights.safetensors (88.75 MB)
  checkpoint-110/optimizer.bin (178.09 MB)
  checkpoint-110/scheduler.bin (0.00 MB)
  checkpoint-220/pytorch_lora_weights.safetensors (88.75 MB)
  checkpoint-220/optimizer.bin (178.09 MB)
  checkpoint-220/scheduler.bin (0.00 MB)
  checkpoint-330/pytorch_lora_weights.safetensors (88.75 MB)
  checkpoint-330/optimizer.bin (178.09 MB)
  checkpoint-330/scheduler.bin (0.00 MB)


## 11. Final LoRA Evaluation Generation
Loads the fine-tuned LoRA weights and generates outputs for all 12 evaluation prompts using the exact same seeds as baseline.


In [18]:
import os
import pandas as pd
import torch
from diffusers import StableDiffusionXLPipeline, AutoencoderKL

print("Loading SDXL pipeline with trained Brutalism LoRA...")

vae = AutoencoderKL.from_pretrained(VAE_ID, torch_dtype=torch.float16)
lora_pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_ID,
    vae=vae,
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True
).to("cuda")

# Load trained LoRA weights
lora_pipe.load_lora_weights(FINAL_DIR)
lora_pipe.enable_attention_slicing()
print("LoRA weights successfully loaded into pipeline!")

eval_records = []
for idx, prompt_text in enumerate(eval_lines):
    clean_prompt = prompt_text.split(". ", 1)[-1] if ". " in prompt_text[:4] else prompt_text
    seed = SEED + idx
    generator = torch.Generator(device="cuda").manual_seed(seed)

    print(f"Generating LoRA [{idx+1}/{len(eval_lines)}] Seed={seed}: {clean_prompt[:60]}...")
    image = lora_pipe(
        prompt=clean_prompt,
        num_inference_steps=30,
        guidance_scale=7.5,
        generator=generator,
        height=RESOLUTION,
        width=RESOLUTION
    ).images[0]

    out_name = f"lora_eval_{idx+1:02d}.jpg"
    out_path = os.path.join(LORA_DIR, out_name)
    image.save(out_path, "JPEG", quality=95)

    eval_records.append({
        "prompt_id": f"LORA_EVAL_{idx+1:02d}",
        "prompt": clean_prompt,
        "seed": seed,
        "output_path": out_path
    })

eval_df = pd.DataFrame(eval_records)
eval_df.to_csv(os.path.join(OUTPUT_DIR, "evaluation_results.csv"), index=False)
print(f"Saved {len(eval_records)} LoRA evaluation images to {LORA_DIR}")



Loading SDXL pipeline with trained Brutalism LoRA...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

...

Generating LoRA [11/12] Seed=52: Brutalist government pavilion in direct high-noon desert day...


  0%|          | 0/30 [00:00<?, ?it/s]

Generating LoRA [12/12] Seed=53: Brutalist archive building under diffuse overcast sky, showi...


  0%|          | 0/30 [00:00<?, ?it/s]

Saved 12 LoRA evaluation images to /content/drive/MyDrive/Architectural-Styles-LoRA/outputs/brutalism_sdxl_lora_v1/lora


In [19]:
from PIL import Image
import matplotlib.pyplot as plt
import glob
import os
import math

lora_dir = "/content/drive/MyDrive/Architectural-Styles-LoRA/outputs/brutalism_sdxl_lora_v1/lora"

images = sorted(
    glob.glob(os.path.join(lora_dir, "*.png")) +
    glob.glob(os.path.join(lora_dir, "*.jpg")) +
    glob.glob(os.path.join(lora_dir, "*.jpeg"))
)

print(f"Found {len(images)} LoRA evaluation images")

cols = 3
rows = math.ceil(len(images) / cols)

fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows))
axes = axes.flatten()

for ax, img_path in zip(axes, images):
    img = Image.open(img_path)
    ax.imshow(img)
    ax.set_title(os.path.basename(img_path), fontsize=10)
    ax.axis("off")

# Hide unused cells
for ax in axes[len(images):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

Output hidden; open in https://colab.research.google.com to view.

## 12. Style Leakage Probing
Tests whether the model inappropriately injects Gothic arches, Classical pediments, Baroque ornament, or Postmodern motifs.


In [22]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive/Architectural-Styles-LoRA"):
    for file in files:
        if "leak" in file.lower() or "prompt" in file.lower():
            print(os.path.join(root, file))

/content/drive/MyDrive/Architectural-Styles-LoRA/evaluation/brutalism_prompts.txt
/content/drive/MyDrive/Architectural-Styles-LoRA/evaluation/style_leakage_prompts.txt


In [25]:
leakage_prompts_file = "/content/drive/MyDrive/Architectural-Styles-LoRA/evaluation/style_leakage_prompts.txt"

print("Using:", leakage_prompts_file)
print("Exists:", os.path.exists(leakage_prompts_file))

Using: /content/drive/MyDrive/Architectural-Styles-LoRA/evaluation/style_leakage_prompts.txt
Exists: True


In [26]:
LEAKAGE_DIR = os.path.join(OUTPUT_DIR, "leakage_tests")
os.makedirs(LEAKAGE_DIR, exist_ok=True)

with open(leakage_prompts_file, "r", encoding="utf-8") as f:
    leak_lines = [line.strip() for line in f if line.strip() and not line.startswith("#") and "[" in line]

leakage_records = []
for idx, line in enumerate(leak_lines):
    tag, prompt_text = line.split("] ", 1)
    tag = tag.replace("[", "")
    seed = SEED + 100 + idx
    generator = torch.Generator(device="cuda").manual_seed(seed)

    print(f"Testing Leakage [{idx+1}/{len(leak_lines)}] {tag}...")
    image = lora_pipe(
        prompt=prompt_text,
        num_inference_steps=30,
        guidance_scale=7.5,
        generator=generator,
        height=RESOLUTION,
        width=RESOLUTION
    ).images[0]

    out_name = f"leakage_test_{idx+1:02d}_{tag}.jpg"
    out_path = os.path.join(LEAKAGE_DIR, out_name)
    image.save(out_path, "JPEG", quality=95)

    leakage_records.append({
        "test_tag": tag,
        "prompt": prompt_text,
        "seed": seed,
        "output_path": out_path
    })

leak_df = pd.DataFrame(leakage_records)
leak_df.to_csv(os.path.join(OUTPUT_DIR, "leakage_results.csv"), index=False)
print(f"Saved {len(leakage_records)} leakage test images to {LEAKAGE_DIR}")



Testing Leakage [1/8] 1. Baseline Test...


  0%|          | 0/30 [00:00<?, ?it/s]

Testing Leakage [2/8] 2. Anti-Gothic Test...


  0%|          | 0/30 [00:00<?, ?it/s]

Testing Leakage [3/8] 3. Anti-Classical & Baroque Test...


...

Testing Leakage [7/8] 7. Anti-Art Nouveau Test...


  0%|          | 0/30 [00:00<?, ?it/s]

Testing Leakage [8/8] 8. Material Contrast Test...


  0%|          | 0/30 [00:00<?, ?it/s]

Saved 8 leakage test images to /content/drive/MyDrive/Architectural-Styles-LoRA/outputs/brutalism_sdxl_lora_v1/leakage_tests


In [27]:
from PIL import Image
import matplotlib.pyplot as plt
import glob
import os
import math

leakage_dir = "/content/drive/MyDrive/Architectural-Styles-LoRA/outputs/brutalism_sdxl_lora_v1/leakage_tests"

images = sorted(
    glob.glob(os.path.join(leakage_dir, "*.jpg")) +
    glob.glob(os.path.join(leakage_dir, "*.png"))
)

print(f"Found {len(images)} leakage test images")

cols = 2
rows = math.ceil(len(images) / cols)

fig, axes = plt.subplots(rows, cols, figsize=(12, 6 * rows))
axes = axes.flatten()

for ax, img_path in zip(axes, images):
    img = Image.open(img_path)

    ax.imshow(img)
    ax.set_title(os.path.basename(img_path), fontsize=11)
    ax.axis("off")

for ax in axes[len(images):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

Output hidden; open in https://colab.research.google.com to view.

## 13. Memorization vs. Generalization Inspection
Computes feature similarity between generated evaluation images and the 22 training images using ResNet-34.


In [28]:
import torchvision.models as models
import torchvision.transforms as transforms
import numpy as np

print("=== Running Memorization Check ===")
model = models.resnet34(weights=models.ResNet34_Weights.DEFAULT).to("cuda")
feat_extractor = torch.nn.Sequential(*(list(model.children())[:-1]))
feat_extractor.eval()

transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Extract embeddings for 22 training images
train_embs = {}
with torch.no_grad():
    for _, r in brut_train.iterrows():
        fpath = os.path.join(IMAGES_DIR, f"{r['image_id']}.jpg")
        with Image.open(fpath) as im:
            t = transform(im.convert("RGB")).unsqueeze(0).to("cuda")
            e = feat_extractor(t).squeeze().cpu().numpy()
            train_embs[r['image_id']] = e / (np.linalg.norm(e) + 1e-9)

# Compare each generated LoRA image against training images
mem_records = []
with torch.no_grad():
    for r in eval_records:
        gen_path = r["output_path"]
        with Image.open(gen_path) as im:
            t = transform(im.convert("RGB")).unsqueeze(0).to("cuda")
            e = feat_extractor(t).squeeze().cpu().numpy()
            gen_emb = e / (np.linalg.norm(e) + 1e-9)

        best_sim = -1.0
        best_id = ""
        for tr_id, tr_emb in train_embs.items():
            sim = float(np.dot(gen_emb, tr_emb))
            if sim > best_sim:
                best_sim = sim
                best_id = tr_id

        flag = "HIGH_SIMILARITY" if best_sim >= 0.85 else ("MODERATE_SIMILARITY" if best_sim >= 0.70 else "NOVEL_COMPOSITION")
        mem_records.append({
            "generated_image": os.path.basename(gen_path),
            "closest_training_image": best_id,
            "similarity": round(best_sim, 4),
            "flag": flag
        })

mem_df = pd.DataFrame(mem_records)
mem_df.to_csv(os.path.join(OUTPUT_DIR, "memorization_results.csv"), index=False)
print("Memorization inspection complete. Results:")
print(mem_df.to_string(index=False))



=== Running Memorization Check ===
Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 144MB/s]


Memorization inspection complete. Results:
 generated_image closest_training_image  similarity                flag
lora_eval_01.jpg               ARC_0020      0.8041 MODERATE_SIMILARITY
lora_eval_02.jpg               ARC_0017      0.7505 MODERATE_SIMILARITY
lora_eval_03.jpg               ARC_0018      0.8581     HIGH_SIMILARITY
lora_eval_04.jpg               ARC_0012      0.7908 MODERATE_SIMILARITY
lora_eval_05.jpg               ARC_0012      0.7804 MODERATE_SIMILARITY
lora_eval_06.jpg               ARC_0018      0.8541     HIGH_SIMILARITY
lora_eval_07.jpg               ARC_0010      0.7528 MODERATE_SIMILARITY
lora_eval_08.jpg               ARC_0011      0.6017   NOVEL_COMPOSITION
lora_eval_09.jpg               ARC_0015      0.7630 MODERATE_SIMILARITY
lora_eval_10.jpg               ARC_0006      0.7509 MODERATE_SIMILARITY
lora_eval_11.jpg               ARC_0018      0.7189 MODERATE_SIMILARITY
lora_eval_12.jpg               ARC_0006      0.7863 MODERATE_SIMILARITY


## 14. Package Results for Download
Compresses outputs into a zip archive on Google Drive for easy local inspection.


In [29]:
import shutil

zip_filename = os.path.join(PROJECT_ROOT, f"{EXPERIMENT_NAME}_results.zip")
print(f"Creating archive: {zip_filename}...")
shutil.make_archive(zip_filename.replace(".zip", ""), 'zip', OUTPUT_DIR)
print(f"SUCCESS: Results packaged at {zip_filename}")
print(f"File size: {os.path.getsize(zip_filename) / (1024*1024):.2f} MB")



Creating archive: /content/drive/MyDrive/Architectural-Styles-LoRA/brutalism_sdxl_lora_v1_results.zip...
SUCCESS: Results packaged at /content/drive/MyDrive/Architectural-Styles-LoRA/brutalism_sdxl_lora_v1_results.zip
File size: 821.79 MB
